In [1]:
!pip install sentence-transformers faiss-cpu groq gTTS openai-whisper edge-tts -q
!pip install edge-tts nest_asyncio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 8.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.27.2 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [2]:
import numpy as np
import pandas as pd
import faiss
import asyncio
import time
from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import userdata, files
from gtts import gTTS
from IPython.display import Audio
import whisper

In [7]:
with open("maju_dataset_cleaned.txt", "r", encoding="utf-8") as file:
    data = file.read()

print(data[:500])

MAJU University Information Dataset

Muhammad Ali Jinnah University (MAJU) is located in Karachi, Pakistan.

MAJU offers undergraduate, graduate, and postgraduate degree programs.

The Faculty of Computing offers BS Computer Science, BS Artificial Intelligence, and BS Software Engineering programs.

Admissions at MAJU usually open in Spring and Fall semesters.

Students can apply online through the MAJU admission portal.

MAJU provides merit-based and need-based scholarships.

The university has


## Chunking and Text Preprocessing

In [8]:
def create_chunks(text, max_chars=600): # Urdu ke liye 600-700 character behtareen hain
    # Pehle double new lines par split karein
    paragraphs = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 10]

    chunks = []
    current_chunk = ""

    for para in paragraphs:
        # Agar single paragraph hi max_chars se bara hai to usay safely handle karein
        if len(para) > max_chars:
            if current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = ""
            # Paragraph ko chote hisson mein break karke chunks mein daal dein
            start = 0
            while start < len(para):
                chunks.append(para[start:start+max_chars].strip())
                start += max_chars
            continue

        # Agar naya paragraph mojooda chunk mein fit aa sakta hai
        if len(current_chunk) + len(para) + 2 <= max_chars:
            current_chunk += para + "\n\n"
        else:
            # Agar fit nahi aata to purani chunk save karein aur nayi shuru karein
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para + "\n\n"

    # Aakhri bachi hui chunk ko add karein
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

# 600 characters ke sath chunks banayein
chunks = create_chunks(data, max_chars=600)

print("Total Chunks:", len(chunks))
print("--- First Chunk Sample ---")
print(chunks[0])

Total Chunks: 62
--- First Chunk Sample ---
MAJU University Information Dataset

Muhammad Ali Jinnah University (MAJU) is located in Karachi, Pakistan.

MAJU offers undergraduate, graduate, and postgraduate degree programs.

The Faculty of Computing offers BS Computer Science, BS Artificial Intelligence, and BS Software Engineering programs.

Admissions at MAJU usually open in Spring and Fall semesters.

Students can apply online through the MAJU admission portal.

MAJU provides merit-based and need-based scholarships.

The university has modern computer labs for students.


## Build Embeddings and FAISS Index

In [9]:
def build_index(model_name):
    model = SentenceTransformer(model_name)

    start = time.time()
    embeddings = model.encode(chunks)
    end = time.time()

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings))

    return model, index, round(end - start, 2)

## Retrieval Models Setup

In [10]:
retrieval_models = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    #"Multilingual MiniLM": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    "BAAI": "BAAI/bge-m3"
}

retrievers = {}

for name, model_name in retrieval_models.items():
    model, index, build_time = build_index(model_name)
    retrievers[name] = {
        "model": model,
        "index": index,
        "build_time": build_time
    }

    print(name, "created in", build_time, "seconds")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (18 outputs left out)

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

BAAI created in 140.44 seconds


## Retrieval Function

In [11]:
# 7. Retrieval Function
def retrieve_context(query, retriever_name="BAAI", top_k=5):
    model = retrievers[retriever_name]["model"]
    index = retrievers[retriever_name]["index"]

    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)

    results = []
    for idx in indices[0]:
        results.append(chunks[idx])

    return "\n\n".join(results)

##### Groq API Setup

In [24]:
from google.colab import userdata

# Fetch the API key securely from Colab Secrets
try:
    api_key = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=api_key)
    print("Groq client successfully initialized using secret key.")
except Exception as e:
    print(f"Error: {e}. Ensure 'GROQ_API_KEY' is set in Colab Secrets.")

Groq client successfully initialized using secret key.


## Language Detection

In [13]:
def detect_language(text):
    for char in text:
        if '\u0600' <= char <= '\u06FF':
            return "ur"
    return "en"

In [14]:
llm_models = {
    "Llama 3.1 8B": "llama-3.1-8b-instant",
    "Llama 3.3 70B": "llama-3.3-70b-versatile"
}

## Answer Generation

In [15]:
def generate_answer(
    query,
    retriever_name="MiniLM",
    llm_name="Llama 3.3 70B"
):

    context = retrieve_context(
        query,
        retriever_name=retriever_name,
        top_k=5
    )

    lang = detect_language(query)

    language_instruction = (
        "Answer only in Urdu."
        if lang == "ur"
        else "Answer only in English."
    )

    prompt = f"""
You are a MAJU university chatbot.

Rules:
1. Answer ONLY from the provided context.
2. Never use your own knowledge.
3. Never guess.
4. If answer is not present in context, say:
   Information not found in the dataset.
5. {language_instruction}
6. Keep answer short and factual.

Context:
{context}

Question:
{query}

Answer:
"""

    response = client.chat.completions.create(
        model=llm_models[llm_name],
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

## Evaluation Questions

In [16]:
test_questions = [
    "Where is MAJU located?",
    "What is the per credit hour fee for BS Computer Science?",
    "What is the application processing fee?",
    "What is the admission fee at MAJU?",
    "What are the eligibility criteria for BS Computer Science?",
    "Does MAJU offer BS Artificial Intelligence?",
    "What scholarships are available at MAJU?",
    "What is the attendance requirement for final examinations?",
    "What is the email address of Maha Irfan?",
    "When are the final examinations starting?",

    "ماجو کہاں واقع ہے؟",
    "بی ایس کمپیوٹر سائنس کی فی کریڈٹ آور فیس کتنی ہے؟",
    "درخواست پراسیسنگ فیس کتنی ہے؟",
    "ماجو کی داخلہ فیس کتنی ہے؟",
    "بی ایس کمپیوٹر سائنس کی اہلیت کیا ہے؟",
    "کیا ماجو بی ایس آرٹیفیشل انٹیلی جنس پیش کرتا ہے؟",
    "ماجو میں کون کون سی اسکالرشپس دستیاب ہیں؟",
    "فائنل امتحان میں بیٹھنے کے لیے کتنی حاضری ضروری ہے؟",
    "مہا عرفان کا ای میل ایڈریس کیا ہے؟",
    "فائنل امتحانات کب شروع ہوں گے؟"
]

In [25]:
def evaluate_pipeline(
    retriever_name="MiniLM",
    llm_name="Llama 3.1 8B"
):
    results = []

    for i, q in enumerate(test_questions, start=1):
        ans = generate_answer(q, retriever_name, llm_name)

        print("=" * 80)
        print("Question", i, ":", q)
        print("Answer:", ans)

        results.append({
            "Question No": i,
            "Question": q,
            "Retriever": retriever_name,
            "LLM": llm_name,
            "Answer": ans
        })

    return pd.DataFrame(results)

results_df = evaluate_pipeline()
results_df

Question 1 : Where is MAJU located?
Answer: MAJU is located at Main Shahrah-e-Faisal, Block-6 PECHS, Karachi, Pakistan.
Question 2 : What is the per credit hour fee for BS Computer Science?
Answer: The tuition fee for BS Computer Science is PKR 10,000 per credit hour.
Question 3 : What is the application processing fee?
Answer: The application processing fee is PKR 3,000.
Question 4 : What is the admission fee at MAJU?
Answer: The admission fee at MAJU is PKR 20,000.
Question 5 : What are the eligibility criteria for BS Computer Science?
Answer: A minimum of 50% marks in Intermediate, DAE, or A-Levels with Mathematics.
Question 6 : Does MAJU offer BS Artificial Intelligence?
Answer: Yes, MAJU offers BS Artificial Intelligence.
Question 7 : What scholarships are available at MAJU?
Answer: Merit-based scholarships, need-based financial assistance, scholarships for siblings, scholarships for alumni, scholarships for differently abled students, and sports scholarships are available at MAJU

,Question No,Question,Retriever,LLM,Answer
0,1,Where is MAJU located?,MiniLM,Llama 3.1 8B,"MAJU is located at Main Shahrah-e-Faisal, Bloc..."
1,2,What is the per credit hour fee for BS Compute...,MiniLM,Llama 3.1 8B,The tuition fee for BS Computer Science is PKR...
2,3,What is the application processing fee?,MiniLM,Llama 3.1 8B,"The application processing fee is PKR 3,000."
3,4,What is the admission fee at MAJU?,MiniLM,Llama 3.1 8B,"The admission fee at MAJU is PKR 20,000."
4,5,What are the eligibility criteria for BS Compu...,MiniLM,Llama 3.1 8B,"A minimum of 50% marks in Intermediate, DAE, o..."
5,6,Does MAJU offer BS Artificial Intelligence?,MiniLM,Llama 3.1 8B,"Yes, MAJU offers BS Artificial Intelligence."
6,7,What scholarships are available at MAJU?,MiniLM,Llama 3.1 8B,"Merit-based scholarships, need-based financial..."
7,8,What is the attendance requirement for final e...,MiniLM,Llama 3.1 8B,Students are required to maintain at least 75%...
8,9,What is the email address of Maha Irfan?,MiniLM,Llama 3.1 8B,maha.irfan@jinnah.edu.
9,10,When are the final examinations starting?,MiniLM,Llama 3.1 8B,Final examinations will be conducted from June...


In [26]:
def evaluate_pipeline(
    retriever_name="BAAI",
    llm_name="Llama 3.3 70B"
):
    results = []

    for i, q in enumerate(test_questions, start=1):
        ans = generate_answer(q, retriever_name, llm_name)

        print("=" * 80)
        print("Question", i, ":", q)
        print("Answer:", ans)

        results.append({
            "Question No": i,
            "Question": q,
            "Retriever": retriever_name,
            "LLM": llm_name,
            "Answer": ans
        })

    return pd.DataFrame(results)

results_df = evaluate_pipeline()
results_df

Question 1 : Where is MAJU located?
Answer: MAJU is located at Main Shahrah-e-Faisal, Block-6 PECHS, Karachi, Pakistan.
Question 2 : What is the per credit hour fee for BS Computer Science?
Answer: PKR 10,000.
Question 3 : What is the application processing fee?
Answer: The application processing fee is PKR 3,000.
Question 4 : What is the admission fee at MAJU?
Answer: The admission fee is PKR 20,000.
Question 5 : What are the eligibility criteria for BS Computer Science?
Answer: A minimum of 50% marks in Intermediate, DAE, or A-Levels with Mathematics.
Question 6 : Does MAJU offer BS Artificial Intelligence?
Answer: Yes.
Question 7 : What scholarships are available at MAJU?
Answer: MAJU offers merit-based scholarships, need-based financial assistance, scholarships for siblings, alumni, differently abled students, and sports scholarships.
Question 8 : What is the attendance requirement for final examinations?
Answer: Students are required to maintain at least 75% attendance to be eligi

,Question No,Question,Retriever,LLM,Answer
0,1,Where is MAJU located?,BAAI,Llama 3.3 70B,"MAJU is located at Main Shahrah-e-Faisal, Bloc..."
1,2,What is the per credit hour fee for BS Compute...,BAAI,Llama 3.3 70B,"PKR 10,000."
2,3,What is the application processing fee?,BAAI,Llama 3.3 70B,"The application processing fee is PKR 3,000."
3,4,What is the admission fee at MAJU?,BAAI,Llama 3.3 70B,"The admission fee is PKR 20,000."
4,5,What are the eligibility criteria for BS Compu...,BAAI,Llama 3.3 70B,"A minimum of 50% marks in Intermediate, DAE, o..."
5,6,Does MAJU offer BS Artificial Intelligence?,BAAI,Llama 3.3 70B,Yes.
6,7,What scholarships are available at MAJU?,BAAI,Llama 3.3 70B,"MAJU offers merit-based scholarships, need-bas..."
7,8,What is the attendance requirement for final e...,BAAI,Llama 3.3 70B,Students are required to maintain at least 75%...
8,9,What is the email address of Maha Irfan?,BAAI,Llama 3.3 70B,maha.irfan@jinnah.edu
9,10,When are the final examinations starting?,BAAI,Llama 3.3 70B,"June 29, 2026."


In [27]:
question = "فائنل امتحان میں بیٹھنے کے لیے کتنی حاضری ضروری ہے؟"

answer = generate_answer(question, retriever_name="BAAI",llm_name="Llama 3.1 8B")

print(answer)

حاضری کی شرائط میں کوئی نرمی نہیں ہے۔ 80% حاضری ضروری ہے۔


In [28]:
question = "فائنل امتحان میں بیٹھنے کے لیے کتنی حاضری ضروری ہے؟"

answer = generate_answer(question, retriever_name="BAAI",llm_name="Llama 3.3 70B")

print(answer)

حاضری کی شرائط میں کوئی نرمی نہیں ہے، طلباء کو حتمی امتحانات کے اہل ہونے کے لیے کم از کم 75% حاضری برقرار رکھنے کی ضرورت ہے۔


In [30]:
question ="What is the admission fee of MAJU?"

answer = generate_answer(question)

print(answer)

The admission fee is PKR 20,000.


In [31]:
question ="What is the admission fee of MAJU?"

context = retrieve_context(
    question,
    retriever_name="BAAI",
    top_k=5
)

print(context)

MAJU offers undergraduate, graduate, and doctoral programs.

Undergraduate programs include BS Computer Science, BS Software Engineering, BS Artificial Intelligence, BS Cyber Security, BS Business Computing, BS Psychology, and BS Biotechnology.

Graduate programs include MBA, MS Computer Science, MS Software Engineering, MS Data Science, MS Biotechnology, and MS Bioinformatics.

Doctoral programs include PhD Computer Science, PhD Management Science, and PhD Biosciences.

The application processing fee is PKR 3,000.

The admission fee is PKR 20,000.

MAJU offers merit-based scholarships.

Students with 85% or above in Intermediate may receive a 100% tuition fee scholarship in the first semester.

Students with 80% to 84.9% may receive a 75% tuition fee scholarship in the first semester.

Students with 75% to 79.9% may receive a 50% tuition fee scholarship in the first semester.

MAJU offers need-based financial assistance.

MAJU offers scholarships for siblings.

MAJU offers scholarship

## Text-to-Speech using gTTS

In [34]:
def tts_gtts(text, lang="en", filename="gtts_output.mp3"):
    tts = gTTS(text=text, lang=lang)
    tts.save(filename)
    return Audio(filename)

In [50]:
question = "ماجو کہاں واقع ہے؟"
answer = generate_answer(question, retriever_name="BAAI",llm_name="Llama 3.1 8B")
lang = detect_language(question)

print(answer)

tts_gtts(answer, lang)

ماجو کراچی، پاکستان میں واقع ہے۔


## Text-to-Speech using edge_tts

In [51]:
import edge_tts
from IPython.display import Audio
import nest_asyncio
import asyncio

nest_asyncio.apply()

async def tts_edge_async(text, lang="en", filename="edge_output.mp3"):

    voice = (
        "ur-PK-AsadNeural"
        if lang == "ur"
        else "en-US-AriaNeural"
    )

    communicate = edge_tts.Communicate(text, voice)

    await communicate.save(filename)

    return filename


def tts_edge(text, lang="en", filename="edge_output.mp3"):

    loop = asyncio.get_event_loop()

    loop.run_until_complete(
        tts_edge_async(text, lang, filename)
    )

    return Audio(filename)

In [52]:
tts_edge(answer, lang)

## English Example

## STT Model 1: Whisper Base

In [74]:
whisper_base = whisper.load_model("base")
print("Whisper Base loaded")

Whisper Base loaded


In [67]:
from google.colab import files

uploaded = files.upload()

audio_file = list(uploaded.keys())[0]

result = whisper_base.transcribe(audio_file)

print("Transcribed Text:")
print(result["text"])

Saving Eng-1.mp3 to Eng-1 (3).mp3


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcribed Text:
 Where is Muhammad Ali Jinnah University located?


## STT Model 2: Whisper Large v3 using Groq

In [62]:
def transcribe_groq_whisper(audio_file):
    with open(audio_file, "rb") as file:
        transcription = client.audio.transcriptions.create(
            file=file,
            model="whisper-large-v3"
        )

    return transcription.text

large_text = transcribe_groq_whisper(audio_file)

print("Whisper Large v3 Transcription:")
print(large_text)

Whisper Large v3 Transcription:
 Where is Muhammad Ali Jinnah University located?


### Compare STT Models

In [63]:
stt_df = pd.DataFrame([
    {
        "Model": "Whisper Base",
        "Transcription": result["text"]
    },
    {
        "Model": "Whisper Large v3",
        "Transcription": large_text
    }
])

display(stt_df)

,Model,Transcription
0,Whisper Base,Where is Muhammad Ali Jinnah University located?
1,Whisper Large v3,Where is Muhammad Ali Jinnah University located?


In [64]:
def speech_chatbot(audio_file, stt_model="base", tts_model="gtts"):
    if stt_model == "base":
        result = whisper_base.transcribe(audio_file)
        query = result["text"]
    else:
        query = transcribe_groq_whisper(audio_file)

    answer = generate_answer(query)
    lang = detect_language(query)

    print("User Query:")
    print(query)

    print("\nAnswer:")
    print(answer)

    if tts_model == "gtts":
        return tts_gtts(answer, lang)
    else:
        return tts_edge(answer, lang)

In [65]:
speech_chatbot(audio_file, stt_model="base", tts_model="gtts")

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


User Query:
 Where is Muhammad Ali Jinnah University located?

Answer:
Muhammad Ali Jinnah University (MAJU) is located at Main Shahrah-e-Faisal, Block-6 PECHS, Karachi, Pakistan.


In [66]:
speech_chatbot(audio_file, stt_model="large-v3", tts_model="edge")

User Query:
 Where is Muhammad Ali Jinnah University located?

Answer:
Muhammad Ali Jinnah University (MAJU) is located at Main Shahrah-e-Faisal, Block-6 PECHS, Karachi, Pakistan.


## Urdu Example



## STT Model 1: Whisper Base

In [119]:
from google.colab import files

uploaded = files.upload()

audio_file = list(uploaded.keys())[0]

result = whisper_base.transcribe(audio_file)

print("Transcribed Text:")
print(result["text"])

Saving Urdu-1.mp3 to Urdu-1 (3).mp3


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcribed Text:
 محمد لی جناہ یونی ورستی کا حواقے ہے


## STT Model 2: Whisper Large v3 using Groq

In [120]:
def transcribe_groq_whisper(audio_file):
    with open(audio_file, "rb") as file:
        transcription = client.audio.transcriptions.create(
            file=file,
            model="whisper-large-v3"
        )

    return transcription.text

large_text = transcribe_groq_whisper(audio_file)

print("Whisper Large v3 Transcription:")
print(large_text)

Whisper Large v3 Transcription:
 محمد علی جناہ یونیورسٹی کہاں واقع ہے؟


### Compare STT Models

In [121]:
stt_df = pd.DataFrame([
    {
        "Model": "Whisper Base",
        "Transcription": result["text"]
    },
    {
        "Model": "Whisper Large v3",
        "Transcription": large_text
    }
])

display(stt_df)

,Model,Transcription
0,Whisper Base,محمد لی جناہ یونی ورستی کا حواقے ہے
1,Whisper Large v3,محمد علی جناہ یونیورسٹی کہاں واقع ہے؟


In [122]:
def speech_chatbot(audio_file, stt_model="base", tts_model="gtts"):
    if stt_model == "base":
        result = whisper_base.transcribe(audio_file)
        query = result["text"]
    else:
        query = transcribe_groq_whisper(audio_file)

    answer = generate_answer(query)
    lang = detect_language(query)

    print("User Query:")
    print(query)

    print("\nAnswer:")
    print(answer)

    if tts_model == "gtts":
        return tts_gtts(answer, lang)
    else:
        return tts_edge(answer, lang)

In [123]:
speech_chatbot(audio_file, stt_model="base", tts_model="gtts")

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


User Query:
 محمد لی جناہ یونی ورستی کا حواقے ہے

Answer:
محمد علی جناح یونیورسٹی (MAJU) مین شاہراہ فیصل، بلاک-6 PECHS، کراچی، پاکستان میں واقع ہے۔


In [124]:
speech_chatbot(audio_file, stt_model="large-v3", tts_model="edge")

User Query:
 محمد علی جناہ یونیورسٹی کہاں واقع ہے؟

Answer:
اطلاعات نہیں مل سکیں۔
